In [1]:
import os
import pandas as pd

input_path = '../POLITICS_finetuning/processed_data.csv'
df = pd.read_csv(input_path)

In [2]:
df

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


In [3]:
import torch
import pandas as pd
import csv
from smc_steer_summary import bias_model_factory

bias_model_path = 'absolute/path/to/bias_model'
bias_tokenizer_path = 'absolute/path/to/bias_tokenizer'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

def generate_summaries(tokenizer, model, df, out_path, is_gpt=False, is_llama=False, bias=False):

    if not is_llama:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)

    # ids = []
    # summaries = [[] for _ in range(3)]  # generate 3 summaries per article

    n_summaries = 3

    with open(out_path, 'w') as f:
        writer = csv.writer(f)

        for idx, row in df.iterrows():
            title, text = row.title, row.body

            stances = ['left', 'right', 'center'] if not bias else [row.stance]

            # print(f'{idx}: {id}')
            print(f'{idx}: {title}')

            for stance in stances:

                prompt = f'Summarize this article{" with " + stance + " bias" if bias else ""}: {text}'
                end_prompt = '\nSummary:'

                # llama allows 2048 tokens
                if is_llama:
                    # llama does not need a separate tokenizer object
                    tokens = model.tokenize(prompt.encode('utf-8'), add_bos=True)
                    end_prompt_len = len(model.tokenize(end_prompt.encode('utf-8')))

                    # truncate to context size, accounting for max summary size
                    prompt = model.detokenize(tokens[:model.n_ctx()-512-end_prompt_len]).decode('utf-8')
                    prompt += end_prompt

                    print(prompt)
                    print(len(model.tokenize(prompt.encode('utf-8'))))
                    
                    for i in range(n_summaries):
                        summary = ''
                        while summary == '':
                            output = model(
                                prompt,
                                max_tokens=512,
                                top_p=0.95,
                                top_k=50,
                                echo=False,
                            )

                            summary = output['choices'][0]['text']
                            if summary.strip() != '':
                                break
                            print('>> empty summary, retrying...')
                                
                        print(summary)

                        pred_bias, _ = bias_model(summary)
                        writer.writerow([title, summary, pred_bias, stance])
                else:
                    offset = 0
                    if hasattr(tokenizer, 'model_max_length'):
                        offset = 512 if is_gpt else 0
                        end_prompt_len = tokenizer(end_prompt, return_tensors="pt").input_ids.shape[1]
                        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=tokenizer.model_max_length-offset-end_prompt_len)
                        offset = inputs.input_ids.shape[1] if is_gpt else 0
                        prompt = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
                        inputs = tokenizer(prompt + end_prompt, return_tensors="pt", truncation=True)
                    else:
                        prompt += end_prompt
                        inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

                    inputs = inputs.to(device)

                    summary_ids = model.generate(
                        inputs.input_ids,
                        min_length=10,
                        max_new_tokens=512,
                        do_sample=True,
                        top_k=50,
                        top_p=0.95,
                        num_return_sequences=n_summaries,
                        pad_token_id=tokenizer.eos_token_id
                    )

                    for i, summary_id in enumerate(summary_ids):
                        summary = tokenizer.decode(summary_id[offset:], skip_special_tokens=True)
                        print(summary)
                        # summaries[i].append(summary)

                        pred_bias, _ = bias_model(summary)
                        writer.writerow([title, summary, pred_bias, stance])

            # ids.append(id)

            # save every 30 articles
            # if (idx + 1) % 30 == 0:
            #     save_summaries(summaries, out_path)

def save_summaries(ids, summaries, out_path):
    data = {"id": ids, **{f"summary{i+1}": summaries[i] for i in range(len(summaries))}}
    df = pd.DataFrame(data)
    df.to_csv(out_path, index=False)

def summarize(tokenizer, model, df, out_path, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama, bias=False)

def summarize_with_leaning(tokenizer, model, df, out_path, is_gpt=False, is_llama=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, is_llama, bias=True)

/Users/ellieyhc/Documents/School/23-24/6.8610/project/NLP-Project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# BART

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

In [ ]:
summarize(bart_tokenizer, bart_model, df, '../dataset/bart.csv')

In [ ]:
for leaning in ['left', 'center', 'right']:
    output_path = f'../dataset/bart-{leaning}.csv'
    summarize_with_leaning(bart_tokenizer, bart_model, df, output_path, bias=leaning)

# T5

In [ ]:
from transformers import AutoTokenizer, AutoModelWithLMHead

t5_tokenizer = AutoTokenizer.from_pretrained('t5-base')
t5_model = AutoModelWithLMHead.from_pretrained('t5-base', return_dict=True)

In [ ]:
summarize(t5_tokenizer, t5_model, df, '../dataset/t5.csv')

# GPT2 and GPT Neo

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

In [ ]:
summarize(gpt2_tokenizer, gpt2_model, df, '../dataset/gpt2.csv', is_gpt=True)

In [ ]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

neo_model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
neo_tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")

In [ ]:
summarize(neo_model, neo_tokenizer, df, '../dataset/neo-test.csv', is_gpt=True)

# Llama 2 (TODO)

In [ ]:
from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = LlamaTokenizer.from_pretrained("/output/path")
llama_model = LlamaForCausalLM.from_pretrained("/output/path")

Using `llama.cpp` (with llama-cpp-python bindings) to run this faster. 
- https://github.com/ggerganov/llama.cpp 
- https://github.com/abetlen/llama-cpp-python

In [7]:
import llama_cpp

In [8]:
llama_path = "absolute/path/to/llama"

In [9]:
llm = llama_cpp.Llama(model_path=llama_path, n_ctx=2048)

llama_model_loader: loaded meta data with 16 key-value pairs and 291 tensors from /Users/ellieyhc/Documents/Research/llama.cpp/models/llama-2-7b/ggml-model-q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: - tensor    0:                token_embd.weight q4_0     [  4096, 32000,     1,     1 ]
llama_model_loader: - tensor    1:               output_norm.weight f32      [  4096,     1,     1,     1 ]
llama_model_loader: - tensor    2:                    output.weight q6_K     [  4096, 32000,     1,     1 ]
llama_model_loader: - tensor    3:              blk.0.attn_q.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    4:              blk.0.attn_k.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    5:              blk.0.attn_v.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    6:         blk.0.attn_output.weight q4_0     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    7:            blk.0

In [7]:
output = llm(
  "Q: Name the planets in the solar system? A: ", # Prompt
  max_tokens=32, # Generate up to 32 tokens
  stop=["Q:", "\n"], # Stop generating just before the model would generate a new question
  echo=False # Echo the prompt back in the output
)


llama_print_timings:        load time =     840.28 ms
llama_print_timings:      sample time =       2.43 ms /    26 runs   (    0.09 ms per token, 10699.59 tokens per second)
llama_print_timings: prompt eval time =     840.12 ms /    15 tokens (   56.01 ms per token,    17.85 tokens per second)
llama_print_timings:        eval time =    2346.41 ms /    25 runs   (   93.86 ms per token,    10.65 tokens per second)
llama_print_timings:       total time =    3223.61 ms


In [8]:
print(output)

{'id': 'cmpl-d7414520-f762-49b4-9a01-fce21bd6d129', 'object': 'text_completion', 'created': 1700764923, 'model': '/Users/ellieyhc/Documents/Research/llama.cpp/models/llama-2-7b/ggml-model-q4_0.gguf', 'choices': [{'text': '8 – Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus and Neptune', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 15, 'completion_tokens': 26, 'total_tokens': 41}}


In [10]:
summarize(None, llm, df, 'llama.csv', is_llama=True)

0: Elizabeth Cheney blasted by older sister over gay-marriage stance
Summarize this article: Call it Cheney versus Cheney.
Mary Cheney, one of ex-Vice President Dick Cheney’s two daughters, has taken to Facebook to blast her older sibling, Elizabeth, a Wyoming Senate candidate, for the latter’s stance on same-sex marriage, The New York Times is reporting.
Mary Cheney, openly lesbian and married to Heather Poe since 2012, reportedly posted to her personal page on the social media site: “For the record, I love my sister, but she is dead wrong on the issue of marriage.
“Freedom means freedom for everyone. That means that all families — regardless of how they look or how they are made — all families are entitled to the same rights, privileges and protections as every other.”
The Times reports Liz Cheney on Friday first articulated her position on the controversial subject, saying it should be something for voters to decide on a state-by-state basis, and not a matter for “judges” or “legisl


llama_print_timings:        load time =   25703.93 ms
llama_print_timings:      sample time =       2.85 ms /    30 runs   (    0.10 ms per token, 10518.93 tokens per second)
llama_print_timings: prompt eval time =   25703.28 ms /   444 tokens (   57.89 ms per token,    17.27 tokens per second)
llama_print_timings:        eval time =    3045.26 ms /    29 runs   (  105.01 ms per token,     9.52 tokens per second)
llama_print_timings:       total time =   28792.69 ms
Llama.generate: prefix-match hit


 In her bid to unseat longtime Senator Mike Enzi in Wyoming, Liz Cheney is under fire from within her own family for not being supportive enough of same sex marriage rights. Her sister Mary has publicly come out against the idea.



llama_print_timings:        load time =   25703.93 ms
llama_print_timings:      sample time =       5.20 ms /    54 runs   (    0.10 ms per token, 10382.62 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    5442.67 ms /    54 runs   (  100.79 ms per token,     9.92 tokens per second)
llama_print_timings:       total time =    5519.39 ms
Llama.generate: prefix-match hit


 Call it Cheney versus Cheney.
1: Mary Cheney: Sister Is 'Dead Wrong' On Gay Marriage
Summarize this article: Mary Cheney, the younger sister of Wyoming U.S Senate candidate Liz Cheney (R) and daughter of former Vice President Dick Cheney, criticized her sister's opposition to same-sex marriage in a Friday Facebook post, the New York Times reports.
According to the Times, Mary Cheney, who is openly gay and married her partner, Heather Poe, in Washington, DC last year, took to Facebook to rebuke her sister's remarks.
“For the record, I love my sister, but she is dead wrong on the issue of marriage," she wrote.
"Freedom means freedom for everyone," she continued. "That means that all families — regardless of how they look or how they are made — all families are entitled to the same rights, privileges and protections as every other."
On Friday, Liz Cheney issued a statement voicing her opposition to same-sex marriage.
"I am strongly pro-life and I am not pro-gay marriage," she said. “I be

...


llama_print_timings:        load time =   25703.93 ms
llama_print_timings:      sample time =       3.35 ms /    36 runs   (    0.09 ms per token, 10749.48 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    3892.63 ms /    36 runs   (  108.13 ms per token,     9.25 tokens per second)
llama_print_timings:       total time =    3947.36 ms
Llama.generate: prefix-match hit


 With the U.S. Senate seat left vacant by President Barack Obama’s nomination of Secretary of State John Kerry now up for grabs in a special election, Massachusetts Gov. Deval Patrick on Wednesday picked a former chief of staff to fill Sen. Kerry's open seat until a special election in June.
The Democratic governor picked former Chief of Staff William "Mo" Cowan who also served as Patrick’s chief legal counsel.
18: Mary Cheney Criticizes Her Sister on Same-Sex Marriage
Summarize this article: Mary Cheney, the younger sister of Liz Cheney, a Wyoming Senate candidate, sharply criticized her sister’s stance on same-sex marriage and urged her own Facebook friends to share the message.
Posting on Facebook on Friday evening, Mary Cheney, who is gay and married her longtime partner last year, wrote: “For the record, I love my sister, but she is dead wrong on the issue of marriage.”
Their father, former Vice President Dick Cheney, supports same-sex marriage, and the younger Cheney echoed some 


llama_print_timings:        load time =   25703.93 ms
llama_print_timings:      sample time =       9.25 ms /   100 runs   (    0.09 ms per token, 10806.14 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =   10390.08 ms /   100 runs   (  103.90 ms per token,     9.62 tokens per second)
llama_print_timings:       total time =   10542.31 ms
Llama.generate: prefix-match hit


 Mary Cheney sharply criticized her sister’s stance on same sex marriage and urged her own Facebook friends to share the message.



llama_print_timings:        load time =   25703.93 ms
llama_print_timings:      sample time =       2.75 ms /    30 runs   (    0.09 ms per token, 10901.16 tokens per second)
llama_print_timings: prompt eval time =   36068.72 ms /   624 tokens (   57.80 ms per token,    17.30 tokens per second)
llama_print_timings:        eval time =    4785.29 ms /    29 runs   (  165.01 ms per token,     6.06 tokens per second)
llama_print_timings:       total time =   40901.33 ms
Llama.generate: prefix-match hit


In [ ]:
df = pd.read_csv('llama.csv')
df

In [ ]:
output_path = f'../dataset/llama-prompt.csv'
summarize_with_leaning(llama_tokenizer, llama_model, df, output_path)